In [1]:
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download("punkt_tab")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [2]:
def preprocess(text):
    tokens = word_tokenize(text.lower())
    lemmatizer = WordNetLemmatizer()
    tokens = [
        lemmatizer.lemmatize(w)
        for w in tokens
        if w.isalpha()
    ]
    return tokens

In [3]:
def build_vocab(tokens):
    vocab = list(set(tokens))
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for i, w in enumerate(vocab)}
    return vocab, word2idx, idx2word

In [4]:
def one_hot(word, word2idx, size):
    vec = np.zeros(size)
    if word in word2idx:
        vec[word2idx[word]] = 1
    return vec

In [5]:
def create_data(tokens, window=2):
    X, Y = [], []
    for i in range(window, len(tokens) - window):
        context = []
        for j in range(i - window, i + window + 1):
            if j != i:
                context.append(tokens[j])
        X.append(context)
        Y.append(tokens[i])
    return X, Y

In [6]:
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

In [7]:
def forward(x, W1, W2):
    h = np.dot(x, W1)      # Hidden = Embedding
    y = np.dot(h, W2)     # Output
    return h, softmax(y)

In [8]:
def backward(x, h, y, t, W2):
    error = y - t
    dW2 = np.outer(h, error)
    dW1 = np.outer(x, np.dot(W2, error))
    return dW1, dW2

In [9]:
def train(X, Y, word2idx, size, embed=10, lr=0.01, epochs=500):
    W1 = np.random.rand(size, embed)   # Embedding Matrix
    W2 = np.random.rand(embed, size)
    for e in range(epochs):
        loss = 0
        for context, target in zip(X, Y):
            x = np.mean([
                one_hot(w, word2idx, size)
                for w in context
            ], axis=0)
            t = one_hot(target, word2idx, size)
            h, y = forward(x, W1, W2)
            loss += -np.sum(t * np.log(y + 1e-9))
            dW1, dW2 = backward(x, h, y, t, W2)
            W1 -= lr * dW1
            W2 -= lr * dW2
        if e % 100 == 0:
            print("Epoch:", e, "Loss:", round(loss, 3))
    return W1, W2

In [10]:
def get_embeddings(W1, vocab):
    embeddings = {}
    for i, word in enumerate(vocab):
        embeddings[word] = W1[i]
    return embeddings


In [11]:
def predict(context, W1, W2, word2idx, idx2word, size):
    x = np.mean([
        one_hot(w, word2idx, size)
        for w in context
    ], axis=0)
    _, y = forward(x, W1, W2)
    return idx2word[np.argmax(y)]


In [21]:
def main():
    corpus = """
    Python is a popular programming language.
    Machine learning models are trained with data.
    Artificial intelligence is rapidly advancing.
    Deep learning uses neural networks.
    Data science involves statistics and coding.
    Algorithms are fundamental to computer science.
    Cloud computing offers scalable solutions.
    """
    # Preprocess
    tokens = preprocess(corpus)
    print("Tokens:", tokens)
    # Vocabulary
    vocab, word2idx, idx2word = build_vocab(tokens)
    size = len(vocab)
    # CBOW Data
    X, Y = create_data(tokens)
    # Train
    W1, W2 = train(X, Y, word2idx, size)
    # Get embeddings
    embeddings = get_embeddings(W1, vocab)
    print("\nWord Embedding Example:")
    print("python →", embeddings["python"])
    # Test prediction
    test = ["machine", "learning"]
    result = predict(test, W1, W2, word2idx, idx2word, size)
    print("\nContext:", test)
    print("Predicted Word:", result)

In [22]:
if __name__ == "__main__":
    main()

Tokens: ['python', 'is', 'a', 'popular', 'programming', 'language', 'machine', 'learning', 'model', 'are', 'trained', 'with', 'data', 'artificial', 'intelligence', 'is', 'rapidly', 'advancing', 'deep', 'learning', 'us', 'neural', 'network', 'data', 'science', 'involves', 'statistic', 'and', 'coding', 'algorithm', 'are', 'fundamental', 'to', 'computer', 'science', 'cloud', 'computing', 'offer', 'scalable', 'solution']
Epoch: 0 Loss: 131.502
Epoch: 100 Loss: 111.287
Epoch: 200 Loss: 84.246
Epoch: 300 Loss: 51.191
Epoch: 400 Loss: 26.411

Word Embedding Example:
python → [ 0.20969167  0.72952233  0.88882485  0.24375363  0.05466588 -0.25057677
  1.39023454 -0.09616827  0.36927628  0.74968419]

Context: ['machine', 'learning']
Predicted Word: language
